### Practice: Parameter Efficient Fine-Tuning
In this notebook, you're gonna fine-tune large language models within limited GPU memory.

In [ ]:
%pip install -U -q peft #==0.17.1
%pip install datasets #==4.0.0
%pip install trl # ==0.9.2
%pip install -U bitsandbytes transformers #==4.45.2

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import datasets

import transformers
from tqdm.auto import tqdm, trange
assert torch.cuda.is_available(), "you need cuda for this part"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
model_name = 'unsloth/Qwen3-8B-Base-bnb-4bit'  # bnb-4bit - quantization type

tokenizer = transformers.AutoTokenizer.from_pretrained(model_name, device_map=device)
tokenizer.pad_token_id = tokenizer.eos_token_id

In [4]:
# the main model weights are loaded in 4-bit precision - but we can still tune LoRAs
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map='auto', low_cpu_mem_usage=True, offload_state_dict=True,
    load_in_4bit=True, torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
)
for param in model.parameters():
    param.requires_grad=False

model.gradient_checkpointing_enable()  # only store a small subset of activations, re-compute the rest.
model.enable_input_require_grads()     # override an implementation quirk in gradient checkpoints that disables backprop unless inputs require grad
# more on gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


### Prompt tuning: the story of a fox (1 point)

![img](https://i.imgur.com/Ux3qQAu.png)

(source: theodd1souts.fandom.com)

In [5]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(10):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))


Output: A quick brown fox jumps over the lazy dog. The quick brown fox


What a blatant lie! This particular fox assures you that it didn't in fact jump over the lazy dog. No, sir! The fox was just minding its own business. __Your task is to train the model to say truth: no dog was jumped over today.__

In [6]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Loss:", loss)

Loss: tensor(3.5380, device='cuda:0', grad_fn=<NllLossBackward0>)


Except, we can't train the entire model - that would be 28GB gradients in float32. Instead, let's run [prompt tuning](https://arxiv.org/abs/2104.08691).

![img](https://i.imgur.com/VwNNKnb.png)


In [7]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    To perform prompt tuning, you will need to replace model's original word embeddings with a layer - THIS layer
     - that inserts trainable prompts instead of the first N token embeddings. """

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True)

    def forward(self, input_ids: torch.LongTensor):
        # input_ids shape: [batch_size, seq length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), "don't forget to prepend several BOS tokens to input_ids"

        # Your task: embed input_ids, but replace the first :num_prompts: tokens with self.learnable_prompts
        # This is because we will prepend :num_prompts: padding tokens at the beginning

        # After you are done, you must produce a word embedding vector for each token in input_ids,
        # except that the first :num_prompts: vectors should equal learnable_prompts;
        # any additional vectors after first :num_prompts: ones should be embedded as usual
        # Note: since you're dealing with trainable params, please torch.cat instead of item assignment

        emb = self.original_word_embeddings(input_ids)
        prompts = self.learnable_prompts.expand(input_ids.shape[0], -1, -1)
        outputs = torch.cat([prompts, emb[:, self.num_prompts:, :]], dim=1)

        return outputs

In [8]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.amp.autocast('cuda'):
    test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
print("Looks legit!")

Looks legit!


__Now that it works,__ let's inject learnable prompts into the main model and teach it about foxes.

In [9]:
assert isinstance(model.model.embed_tokens, nn.Embedding), "you have already replaced the embedding layer. If the replacement is broken, please reload the model"

model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

In [10]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

outputs = model(**batch)
next_word_logits = outputs.logits[:, num_prompts:-1, :]
true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
print("Loss:", loss)

for step in tqdm(range(200)):
    opt.zero_grad()

    output = model(**batch)
    next_word_logits = output.logits[:, num_prompts:-1, :]
    true_next_tokens = batch['input_ids'][:, num_prompts+1:]

    loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
    loss.backward()

    opt.step()

    if step % 25 == 0:
        print("Loss:", loss)

assert loss.item() <= 0.1
print("Good job!")

Loss: tensor(3.6216, device='cuda:0', grad_fn=<NllLossBackward0>)


  0%|          | 0/200 [00:00<?, ?it/s]

Loss: tensor(3.6216, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: tensor(0.5416, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: tensor(0.0020, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: tensor(0.0005, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: tensor(0.0003, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: tensor(0.0002, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: tensor(0.0002, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: tensor(0.0002, device='cuda:0', grad_fn=<NllLossBackward0>)
Good job!


In [11]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)


for i in range(15):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))

# if you did everything right, the model will deny that the fox jumped over the lazy dog


Output: A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway


### Using HuggingFace PEFT

[`peft`](https://huggingface.co/docs/peft/index) is a transformer's sister library that allows you to apply various __p__arameter __e__fficient __f__ine-__t__uning methods to pre-trained transformers. The library imlements both prompt tuning, prefix tuning, as well as several adapter-based techniques under a common interface:



In [4]:
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map='auto', low_cpu_mem_usage=True, offload_state_dict=True,
    load_in_4bit=True, torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
)
for param in model.parameters():
    param.requires_grad=False

model.gradient_checkpointing_enable()  # only store a small subset of activations, re-compute the rest.
model.enable_input_require_grads()

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


In [5]:
import peft
assert isinstance(model.model.embed_tokens, nn.Embedding), "please reload the model"

peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
model = peft.get_peft_model(model, peft_config)  # note: for most peft methods, this line also modifies model in-place
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters (excluding quantization):", sum(p.numel() for p in model.parameters()))

Trainable parameters: 65536
Total parameters (excluding quantization): 4717917184


In [ ]:
# Your task: optimize the PEFT-wrapped model to achieve next token prediction loss < 0.1, but this time using PEFT
# Please note: you no longer need to prepend PAD tokens, but you still need to skip :num_virtual_tokens: first logits.
# Finally, generate the sentence to make sure that the model learned the truth.

In [6]:
opt = torch.optim.Adam([param for param in model.parameters() if param.requires_grad], lr=1e-2)

the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
num_prompts = batch['input_ids'].size(1)

for step in tqdm(range(200)):
    opt.zero_grad()
    outputs = model(input_ids=batch['input_ids'], attention_mask=batch.get('attention_mask', None), labels=batch['input_ids'])
    loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
    loss.backward()
    opt.step()
    if step % 25 == 0:
        print("Loss:", loss)


  0%|          | 0/200 [00:00<?, ?it/s]

Loss: tensor(3.9767, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: tensor(0.2681, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: tensor(0.0012, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: tensor(0.0002, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: tensor(0.0001, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: tensor(0.0001, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: tensor(8.5711e-05, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: tensor(7.3256e-05, device='cuda:0', grad_fn=<NllLossBackward0>)


In [7]:
assert loss.item() <= 0.1
print("Good job!")

Good job!


In [8]:
model

PeftModelForCausalLM(
  (base_model): Qwen3ForCausalLM(
    (model): Qwen3Model(
      (embed_tokens): Embedding(151936, 4096, padding_idx=151654)
      (layers): ModuleList(
        (0-35): 36 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
            (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
            (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
            (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
            (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
            (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          )
          (mlp): Qwen3MLP(
            (gate_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)
            (up_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)
            (down_proj): Linear4bit(in_features=12288, out_features=4096, bias=False)
            (ac

In [9]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(15):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0, :].cpu().numpy().tolist()))


Output: A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway


### Parameter-efficient finetuning with LoRA (1 point)

When training on more serious tasks, you can use low-rank adapters based on the [LoRA paper](https://arxiv.org/pdf/2106.09685.pdf).

The core idea is to add low-rank adapters __in parallel with existing linear layers,__ like this:
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

In the original LoRA paper, the adapters were only added to attention projection matrices. However, [subsequent works](https://arxiv.org/abs/2305.14314) show that it is useful to adapt FFNs as well. But before we do any training, we need to implement the basic LoRA layer.

In [4]:
# re-load the model to remove any previous PEFT tuners
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map='auto', low_cpu_mem_usage=True, offload_state_dict=True,
    load_in_4bit=True, torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
)
for param in model.parameters():
    param.requires_grad=False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /usr/local/lib/python3.10/dist-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2025-11-17 03:29:11.970200: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-17 03:29:13.332008: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
The `load_in_4bit` and `load_in_8bit` argum

In [5]:
class LoRALayer(nn.Module):
    """Wraps a linear layer with LoRA-like adapter. Wraps an existing OPT linear layer"""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # pre-trained (frozen) linear layer
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

    def forward(self, x):
        model_out = self.module(x)
        adapter_out = x @ self.adapter_A @ self.adapter_B

        return model_out + adapter_out

In [6]:
# test your implementation
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "please check your forward pass"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "some adapter weights have no grad"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "bad grad w.r.t. A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "bad grad w.r.t. B"
# note: bad grad means that your code is different from LoRA paper OR that your code is not autograd-friendly (e.g. no_grad)
del dummy_loss, test_linear, test_adapter
print("All tests passed!")

All tests passed!


### Apply LoRA to the model

The code below applies LoRA adapters on top of Q/K/V linear layers in attention blocks. You may also choose to modify other layers:
* self_attn.o_proj - attention output projection
* mlp.up_proj, mlp.gate_proj, mlp.down_proj - transformer feedforward layers
* lm_head - output LM head

__Note:__ please scroll down for the homework task

In [7]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'DecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) > 0, "Did not add any LoRA layers!"

In [8]:
batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False).to(device)
# test a single training step, make sure we get meaningful gradients
with torch.amp.autocast('cuda', dtype=torch.float32):
    out = model.forward(**batch)
    (out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Grad check successful, well done!")

Grad check successful, well done!


### (example) How to train your model

The example below shows how to train the LoRA adapters on a dummy dataset. You will need to run a _similar_ training task later.

__Note:__ please scroll down for the homework task

In [11]:
data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 lines
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
model._hf_peft_config_loaded = True  # silence a warning from HF trainer

trainer = transformers.Trainer(
    model=model, train_dataset=data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=1,
        # note: if you want larger batch size, increase gradient_accumulation_steps
        warmup_steps=250, max_steps=100, learning_rate=2e-4, fp16=True,
        logging_steps=1, output_dir="outputs", report_to="none",
        save_strategy="no"
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
model.config.use_cache = False
# if you see cache warnings, set `model.config.use_cache = False` to silence them. Please re-enable for inference!

trainer.train()

# NOTE: this is just an example! you do not have to wait for this progressbar to finish :)

Step,Training Loss
1,1.263600
2,0.372800
3,1.652700
4,1.138100
5,0.772400
6,1.677000
7,1.584700
8,1.179100
9,0.626500
10,1.204100


TrainOutput(global_step=100, training_loss=1.0367119807004928, metrics={'train_runtime': 150.828, 'train_samples_per_second': 1.326, 'train_steps_per_second': 0.663, 'total_flos': 656277458595840.0, 'train_loss': 1.0367119807004928, 'epoch': 6.25})

### Final task: *actually* train the model (3 points)

Your task is to fine-tune the model to _generate python code_. Please use the above examples for inspiration. More specifically,

* __dataset:__ use [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) or any other data containing python code. Since you do not need much data for this excercise, it is enough to use just shorter validation subset of `codeparrots`
* __preprocessing:__ select python code based on file extentions (.py)  (may skip in case of codeparrot - it is 100% python)
* __short lines:__ please take the first 512 characters of each line
* __adapter type:__ please use LoRA as defined above __plus at least one of:__
   - extra adapter on lm_head
   - extra adapter on MLP components (mlp.*)
   - trainable input embeddings (requires tweaking memory usage)

* __training:__ you do not have to train to convergence. If all goes well, your model should `.generate` code after 500 steps. Please use batch size of at least 4 (4 x 1 x 512 tokens) using `gradient_accumulation_steps=4`.


Note: the peft library also has LoRA implementation. However, we ask that for this assignment you show at least one complete training run with your own LoRA code.

__Alternative assignment:__ Instead of doing python code, feel free to substitute the task with any other dataset, e.g. your favorite artist or podcast, as long as it's ethical. If you choose your own task, please show examples of what your model learned - or did not learn, akin to the code examples below.

In [9]:
model._hf_peft_config_loaded = True
model.config.use_cache = False

In [10]:
data = datasets.load_dataset('codeparrot/codeparrot-clean-valid', split='train')

data = data.map(lambda samples: tokenizer(samples['content'], truncation=True, max_length=512), batched=True)
data.set_format(type='torch', columns=['input_ids', 'attention_mask'])
data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)

Repo card metadata block was not found. Setting CardData to empty.


In [11]:
for layer in model.model.layers:
    layer.mlp.gate_proj = LoRALayer(layer.mlp.gate_proj, rank=lora_rank)#.to(device)
    layer.mlp.up_proj   = LoRALayer(layer.mlp.up_proj,   rank=lora_rank)#.to(device)
    layer.mlp.down_proj = LoRALayer(layer.mlp.down_proj, rank=lora_rank)#.to(device)

trainable_params = [p for p in model.parameters() if p.requires_grad]

In [12]:
print("Number of trainable parameters:", sum(p.numel() for p in trainable_params))

Number of trainable parameters: 19464192


In [13]:
prompts =  ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch']  # feel free to add a few more that are not 100% assiciated with Python

# <A WHOLE LOT OF YOUR CODE>
# generate baseline samples with the selected prompts before finetuning
# please feel free to use transformers.Trainer (as above) or your custom training code
# after the training concludes, please show examples of text generated by your model. It is expected to look like Python code fragments
# print the generation examples nicely (suggestion: use pandas or HTML) for easier comparison
# note: your LoRA-enhanced model can run generation the same way as the non-trained model (above)

In [14]:
def generate(model, prompt: str, max_len: int = 100) -> str:
    if not prompt or prompt.strip() == "":
        prompt_text = tokenizer.eos_token or " "
    else:
        prompt_text = prompt

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=True,
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_len,
            do_sample=True,
            top_k=10,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [15]:
outputs_init_model = []
for prompt in tqdm(prompts):
    output = generate(model, prompt, max_len=100)
    outputs_init_model.append(output)

100%|██████████| 8/8 [02:32<00:00, 19.10s/it]


In [24]:
training_args = transformers.TrainingArguments(
    output_dir="./lora-python-code",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=500,
    learning_rate=1e-4,
    warmup_steps=50,
    #fp16=True,
    bf16=True,
    logging_steps=100,
    #save_steps=250,
    save_total_limit=1,
    report_to="none",
    save_strategy ="no"
)

In [25]:
trainer = transformers.Trainer(
    model=model,
    args=training_args,
    train_dataset=data,
    data_collator=data_collator,
)

trainer.train()

  1%|          | 3/500 [33:10<91:36:17, 663.54s/it]

 20%|██        | 100/500 [23:04<1:27:21, 13.10s/it]
                                                   
 23%|██▎       | 117/500 [55:12<1:28:23, 13.85s/it]

{'loss': 0.8243, 'grad_norm': 1.3930407762527466, 'learning_rate': 8.911111111111111e-05, 'epoch': 0.01}



 40%|████      | 200/500 [46:11<1:08:44, 13.75s/it]
                                                   
 23%|██▎       | 117/500 [1:18:19<1:28:23, 13.85s/it]A

{'loss': 0.9567, 'grad_norm': 1.3974305391311646, 'learning_rate': 6.688888888888889e-05, 'epoch': 0.01}



 60%|██████    | 300/500 [1:09:16<46:40, 14.00s/it]
                                                     
 23%|██▎       | 117/500 [1:41:24<1:28:23, 13.85s/it]A

...


 80%|████████  | 400/500 [1:32:24<21:29, 12.90s/it]
                                                     
 23%|██▎       | 117/500 [2:04:32<1:28:23, 13.85s/it]A

{'loss': 1.0066, 'grad_norm': 2.2738754749298096, 'learning_rate': 2.2444444444444447e-05, 'epoch': 0.03}



100%|██████████| 500/500 [1:55:46<00:00, 14.14s/it]
                                                     
 23%|██▎       | 117/500 [2:27:54<1:28:23, 13.85s/it]A
                                                     
100%|██████████| 500/500 [1:55:46<00:00, 13.89s/it]t]A

{'loss': 0.9863, 'grad_norm': 1.6493523120880127, 'learning_rate': 2.2222222222222224e-07, 'epoch': 0.03}
{'train_runtime': 6946.3899, 'train_samples_per_second': 0.288, 'train_steps_per_second': 0.072, 'train_loss': 0.9509976348876953, 'epoch': 0.03}


TrainOutput(global_step=500, training_loss=0.9509976348876953, metrics={'train_runtime': 6946.3899, 'train_samples_per_second': 0.288, 'train_steps_per_second': 0.072, 'total_flos': 4.33045338574295e+16, 'train_loss': 0.9509976348876953, 'epoch': 0.03258761996317599})

In [28]:
outputs_trained_model = []
for prompt in prompts:
    out = generate(model, prompt, max_len=100)
    outputs_trained_model.append(out)

In [29]:
# This template helps to compare generated code samples in pretty table form
# feel free to present your work in other forms

from IPython.display import HTML, display
table_template = """<table style="border:1px solid black" >
  <tr>
    <th style="text-align: center; border:1px solid black">PROMPT</th>
    <th style="text-align: center; border:1px solid black">BEFORE</th>
    <th style="text-align: center; border:1px solid black">AFTER</th>
  </tr>
{}
</table>"""

row_template = '''  <tr>
    <td style="width:20%; border:1px solid black"><pre align="left">`{}`</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
  </tr>'''

rows = []

for prompt, out_init_model, out_train_model in zip(prompts, outputs_init_model, outputs_trained_model):
    rows.append(row_template.format(prompt, out_init_model, out_train_model))

display(HTML(table_template.format('\n'.join(rows))))

PROMPT,BEFORE,AFTER
``,WhatWhatWhatWhat What What What What What What What What What What What What What What What What What What What What What What What What What What What What What What What What What What Is The Answer What Is The Answer To The Question What Is The Answer To The Question What Is The Answer To The Question What Is The Answer To The Question What Is The Answer To The Question What Is The Answer To The Question What Is The Answer To The Question What Is The Answer To The Question What Is The,"import class Test: def __init__(self, test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test=test"
`import`,import { T } from '../constants/T' import { I } from '../constants/I' import { A } from '../constants/A' import { M } from '../constants/M' import { R } from '../constants/R' /* eslint-disable */ import { getParameterByName } from '../utils/utils' import { getCookie } from '../utils/utils' import { getLocal } from '../utils/utils' import { setLocal } from '../utils/utils' import { setCookie } from '../utils/utils',"import unittest from .. import util from .. import configutil from .. import util import os import sys import time import tempfile import re class ConfigParserTestCase(unittest.TestCase): def setUp(self): self.config = configutil.ConfigParser() self.config.read(os.path.join(os.path.dirname(__file__), 'test.conf')) def test_basic(self): self.assertEqual(self.config.get(""section1"", ""test1""), ""value1"") self.assertEqual(self.config"
`from`,from \eqref{eq:cost} to be minimized. The problem is convex and therefore can be solved by using standard solvers (e.g. CVXPY\footnote{\url{https://github.com/cvxpy/cvxpy}}). The cost function can be reformulated as \begin{equation}\label{eq:cost_reform} c(\mathbf{\bar{x}}) = \mathbf{\bar{x}}^T \mathbf{,"from django.http import HttpResponse, HttpResponseRedirect from django.shortcuts import render #from django.template.loader import get_template #from django.template import Context from django.template import RequestContext from django.template.loader import render_to_string from django.core.files import File from django.core.mail import send_mail from django.conf import settings from django.core.paginator import Paginator, PageNotAnInteger, EmptyPage from django.core.exceptions import PermissionDenied from django.contrib.auth.models import User, Userprofile from django.contrib"
`while`,"while \(\displaystyle f\) is continuous on the interval \(\displaystyle [a,b]\), it may not be differentiable there. However, if \(\displaystyle f\) is differentiable at some point \(\displaystyle c\) in \(\displaystyle [a,b]\), then by the Mean Value Theorem, we have Thus, if \(\displaystyle f\) is differentiable at some point \(\displaystyle c\) in \(\displaystyle [a,b]\","while 1: try: # Get user input and convert it to integer number = int(input('Enter a number: ')) # Check if number is prime and print the result if number == 2: print('%d is a prime number' % number) else: for divisor in range(2, number): if number % divisor == 0: print('%d is not a prime number' % number) break else:"
`try`,"try [a,b,c,d]=decompose(25.4); [e,f,g]=decompose(30.2); h=decompose(40.5); [k,l,m]=decompose(45.6); n=decompose(51.8); [o,p,q]=decompose(62.7); [r,s,t]=decompose(67.9); %define the function decompose = @(x) [log(x","try { import { get, post } from './fetchUtils'; import { getApiEndpoint } from './env'; import { logError } from '../utils/errorUtils'; // eslint-disable-next-line @typescript-eslint/no-explicit-any const fetch = window.fetch as any; export function postUserLogin(username: string, password: string) { const

If you reach this: congratulations! you've completed everything in this practice session.

If you want to dig deeper, try to implement prompt-tuning (for bonus points!).
You can read more about prompt tuning variants in paper [1](https://arxiv.org/abs/2104.08691) or paper [2](https://arxiv.org/abs/2101.00190). Both versions can be implemented by passing trainable prompts as `model.forward(..., past_key_values=your_prompts)`.



### Read more

* How post-training quantization works: https://arxiv.org/abs/2208.07339
* An overview of running large models: https://huggingface.co/docs/accelerate/package_reference/big_modeling
* A general library for different adapter types: https://adapterhub.ml/


### [extra info] Running other models.

This notebook's code can run with other models of similar size, such as [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b) or [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). However, they will require minor code tweaks:
1. change the model name in `AutoModelForCausalLM.from_pretrained()` __and__ `AutoTokenizer`
2. In the prompt tuning code, change `model.model.embed_tokens` to refer to the target model's word embeddings. Simply `print(model)` to navigate to them.
3. Change code to add Lora layers - specifically where you what the transformer block components, since those components now have different names.